# Phase 4 — FCM-Derived Vsh Curve & Interpretation
## FCM Gradational Lithofacies Classification
**Author:** Kumar Yuvraj (23GG5PE02) | IIT Kharagpur

### The Key Algorithmic Insight

Once centroids are interpreted (sand-like vs shale-like clusters), **Vsh at any depth  
is simply the sum of membership probabilities assigned to the shale-like clusters.**

```
FCM_Vsh[depth d] = Σ  u[j, d]   for all j ∈ SHALE_CLUSTERS
```

This is physically meaningful:  
- A depth with 70% membership in the shale cluster → Vsh = 0.70  
- A depth with 30% shale + 70% sand membership → Vsh = 0.30  
- **No threshold. No GR cutoff. The geometry of the log data itself does the work.**

Steps:
1. Compute FCM_Vsh and add to master DataFrame
2. 4-track log panel for 4 representative wells
3. Transition zone table (% depth with max membership < 0.6) — the CV moment

In [13]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

os.chdir('D:/Projects/fcm-lithofacies')
warnings.filterwarnings('ignore')

OUT_DATA = './outputs/data/'
OUT_FIGS = './outputs/figures/'

# ── Load Phase 3 artefacts ─────────────────────────────────────────────────
u           = np.load(os.path.join(OUT_DATA, 'fcm_membership_matrix.npy'))
hard_labels = np.load(os.path.join(OUT_DATA, 'fcm_hard_labels.npy'))
master      = pd.read_csv(os.path.join(OUT_DATA, 'force2020_preprocessed.csv'),
                          low_memory=False)
with open(os.path.join(OUT_DATA, 'cluster_labels.json')) as f:
    cl_info = json.load(f)
C_OPTIMAL = cl_info['C_OPTIMAL']

print(f'u shape      : {u.shape}')
print(f'master shape : {master.shape}')
print(f'C_OPTIMAL    : {C_OPTIMAL}')

# ── Cluster assignments from Phase 3b centroid analysis ────────────────────
# C0  GR=67.1 RHOB=2.38 NPHI=0.32 → Sandy Shale       SHALE
# C1  GR=45.3 RHOB=2.48 NPHI=0.20 → Silty Sandstone   SAND  ← only sand cluster
# C2  GR=52.8 RHOB=2.01 NPHI=0.50 → Organic Shale     SHALE
# C3  GR=66.2 RHOB=2.20 NPHI=0.41 → Shale             SHALE
# C4  GR=89.0 RHOB=2.55 NPHI=0.26 → Marl              SHALE
# C5  GR=80.1 RHOB=2.04 NPHI=0.49 → Pure Shale        SHALE
# C6  GR=91.5 RHOB=2.42 NPHI=0.35 → Shale             SHALE
SAND_CLUSTER  = 1       # the single reservoir-quality cluster
SHALE_CLUSTERS = [0, 2, 3, 4, 5, 6]

# ── Vsh computation — two approaches kept for comparison ───────────────────
#
# APPROACH A — summed shale membership (standard FCM Vsh formula):
#   Vsh_FCM = Σ u[j] for j in SHALE_CLUSTERS  =  1 - u[1]
#   Problem: with 6/7 clusters shale-type, baseline floor is ~0.85
#   Use for: continuous Vsh curve shape, cross-plots, log display
#
# APPROACH B — sand membership directly (NTG-optimised):
#   Sand_prob = u[1]  (probability of belonging to the sand cluster)
#   Vsh_FCM_B = 1 - u[1]  (identical formula, but we threshold on u[1])
#   NTG cutoff: u[1] >= 0.20 means sample has >20% sand cluster affinity
#   This cutoff is calibrated so NTG_FCM ≈ NTG_GR for the same dataset.
#   Use for: NTG computation, reservoir volume estimation

# Compute both
vsh_fcm_a  = np.clip(1.0 - u[SAND_CLUSTER], 0.0, 1.0)   # = sum of shale clusters
sand_prob  = u[SAND_CLUSTER]                               # u[1] directly

# Add all columns to master
master['FCM_Vsh']    = vsh_fcm_a      # continuous Vsh curve (0–1)
master['Sand_prob']  = sand_prob      # raw sand cluster membership
master['FCM_label']  = hard_labels
master['max_mem']    = u.max(axis=0)

# GR Vsh baseline
GR_CLEAN, GR_SHALE = 30.0, 120.0
master['Vsh_GR'] = np.clip(
    (master['GR'] - GR_CLEAN) / (GR_SHALE - GR_CLEAN), 0.0, 1.0
)

# ── Calibrate NTG threshold on Sand_prob ──────────────────────────────────
# Find the Sand_prob cutoff that gives NTG_FCM closest to NTG_GR
ntg_gr = float((master['Vsh_GR'] <= 0.5).sum() / len(master))
best_cutoff, best_gap = 0.20, 999.0
for cutoff in np.arange(0.05, 0.60, 0.01):
    ntg_test = float((sand_prob >= cutoff).sum() / len(sand_prob))
    gap = abs(ntg_test - ntg_gr)
    if gap < best_gap:
        best_gap, best_cutoff = gap, cutoff

SAND_PROB_NTG_CUTOFF = round(best_cutoff, 2)
ntg_fcm_calibrated   = float((sand_prob >= SAND_PROB_NTG_CUTOFF).sum()
                              / len(sand_prob))

print(f'\nVsh_FCM statistics (1 - u[1]):')
print(master['FCM_Vsh'].describe().round(4))
print(f'\nSand_prob statistics (u[1] directly):')
print(master['Sand_prob'].describe().round(4))
print(f'\nNTG calibration:')
print(f'  NTG_GR  (Vsh_GR ≤ 0.50)              = {ntg_gr:.4f}')
print(f'  Best Sand_prob cutoff                 = {SAND_PROB_NTG_CUTOFF:.2f}')
print(f'  NTG_FCM (Sand_prob ≥ {SAND_PROB_NTG_CUTOFF:.2f})            '
      f'= {ntg_fcm_calibrated:.4f}')
print(f'  Residual gap                          = '
      f'{abs(ntg_fcm_calibrated - ntg_gr):.4f}')

# Save
out_path = os.path.join(OUT_DATA, 'force2020_with_fcm_vsh.csv')
master.to_csv(out_path, index=False)
print(f'\n✓ Saved: {out_path}')
print(f'  Columns: {[c for c in master.columns if "Vsh" in c or "Sand" in c or "FCM" in c]}')

u shape      : (7, 1127735)
master shape : (1127735, 9)
C_OPTIMAL    : 7

Vsh_FCM statistics (1 - u[1]):
count    1.127735e+06
mean     8.728000e-01
std      1.519000e-01
min      7.800000e-03
25%      8.582000e-01
50%      9.292000e-01
75%      9.642000e-01
max      9.996000e-01
Name: FCM_Vsh, dtype: float64

Sand_prob statistics (u[1] directly):
count    1.127735e+06
mean     1.272000e-01
std      1.519000e-01
min      4.000000e-04
25%      3.580000e-02
50%      7.080000e-02
75%      1.418000e-01
max      9.922000e-01
Name: Sand_prob, dtype: float64

NTG calibration:
  NTG_GR  (Vsh_GR ≤ 0.50)              = 0.5776
  Best Sand_prob cutoff                 = 0.06
  NTG_FCM (Sand_prob ≥ 0.06)            = 0.5720
  Residual gap                          = 0.0056

✓ Saved: ./outputs/data/force2020_with_fcm_vsh.csv
  Columns: ['FCM_Vsh', 'Sand_prob', 'FCM_label', 'Vsh_GR']


In [8]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 2 — Multi-well 4-track log panel
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# WHY 4 representative wells with different lithology profiles:
# A single well might be 95% shale — the FCM Vsh would be uniformly high and
# uninformative.  Showing one sandstone-dominated, one shale-dominated, one
# mixed, and one carbonate-rich well demonstrates that FCM Vsh behaves
# correctly across all geological environments in the dataset.

LITHO_COLOURS = {
    'Sandstone':       '#DDBB00', 'Sandstone/Shale': '#FF8C00',
    'Shale':           '#707070', 'Marl':             '#228B22',
    'Dolomite':        '#1E90FF', 'Limestone':        '#87CEEB',
    'Chalk':           '#C8A000', 'Coal':             '#1a1a1a',
    'Halite':          '#FF69B4', 'Anhydrite':        '#9370DB',
    'Tuff':            '#A0522D', 'Basement':         '#8B0000',
}

CLUSTER_CMAP = plt.cm.get_cmap('tab10', C_OPTIMAL)

def select_representative_wells(master, n=4):
    """Pick wells dominated by different lithology types."""
    if 'LITH_LABEL' not in master.columns:
        return master['WELL'].unique()[:n].tolist()

    targets = ['Sandstone', 'Shale', 'Limestone', 'Dolomite']
    selected = []
    used = set()
    for target in targets:
        wells = (
            master[master['LITH_LABEL'] == target]
            .groupby('WELL')
            .size()
            .sort_values(ascending=False)
            .index
        )
        for w in wells:
            if w not in used:
                selected.append(w)
                used.add(w)
                break
    # Fill remaining with largest wells
    if len(selected) < n:
        for w in master.groupby('WELL').size().sort_values(ascending=False).index:
            if w not in used:
                selected.append(w)
                used.add(w)
            if len(selected) == n:
                break
    return selected[:n]

rep_wells = select_representative_wells(master, n=4)
print(f'Representative wells selected: {rep_wells}')

def plot_log_panel(wdf, well_name):
    """
    4-track log panel:
      Track 1: GR  — fill between clean (50) and shale (120) baselines
      Track 2: RHOB + NPHI overlay  — standard crossover plot
      Track 3: FCM_Vsh (filled, colour-coded 0→1)
      Track 4: FCM hard label colour strip
    """
    wdf = wdf.sort_values('DEPTH_MD').reset_index(drop=True)
    # Sub-sample long wells
    if len(wdf) > 4000:
        wdf = wdf.iloc[::len(wdf)//4000].reset_index(drop=True)

    depth = wdf['DEPTH_MD'].values

    fig = plt.figure(figsize=(16, 14))
    gs  = GridSpec(1, 4, figure=fig, wspace=0.08)

    # ── Track 1: GR ─────────────────────────────────────────────────────
    ax1 = fig.add_subplot(gs[0, 0])
    gr  = wdf['GR'].values
    ax1.plot(gr, depth, 'k-', lw=0.6)
    ax1.fill_betweenx(depth, 50, gr,
                      where=(gr > 50), color='#C0C0C0', alpha=0.7,
                      label='Shale (GR>50)')
    ax1.fill_betweenx(depth, 0, np.minimum(gr, 50),
                      color='#90EE90', alpha=0.8, label='Sand (GR≤50)')
    ax1.axvline(50,  color='green', ls='--', lw=1.0, alpha=0.7)
    ax1.axvline(120, color='gray',  ls='--', lw=1.0, alpha=0.7)
    ax1.set_xlim(0, 200)
    ax1.set_xlabel('GR (API)', fontsize=9)
    ax1.set_ylabel('Depth (m)', fontsize=10)
    ax1.set_title('Gamma Ray', fontsize=9, fontweight='bold')
    ax1.invert_yaxis()
    ax1.grid(axis='x', alpha=0.25)
    ax1.legend(fontsize=7, loc='upper right')

    # ── Track 2: RHOB + NPHI crossover ──────────────────────────────────
    ax2 = fig.add_subplot(gs[0, 1], sharey=ax1)
    if 'RHOB' in wdf.columns:
        ax2.plot(wdf['RHOB'].values, depth, 'r-', lw=1.0, label='RHOB (g/cc)')
    if 'NPHI' in wdf.columns:
        # Plot NPHI on reversed scale (right axis) to show classic crossover
        ax2b = ax2.twiny()
        ax2b.plot(wdf['NPHI'].values * 100, depth, 'b-', lw=1.0,
                  label='NPHI (%)')
        ax2b.set_xlim(45, -15)   # reversed: high NPHI = left
        ax2b.set_xlabel('NPHI (%)', fontsize=8, color='blue')
        ax2b.tick_params(axis='x', colors='blue', labelsize=7)
    ax2.set_xlim(1.65, 2.85)
    ax2.set_xlabel('RHOB (g/cc)', fontsize=9, color='red')
    ax2.set_title('RHOB + NPHI\n(crossover plot)', fontsize=9, fontweight='bold')
    ax2.tick_params(axis='x', colors='red', labelsize=7)
    ax2.grid(axis='x', alpha=0.25)
    ax2.tick_params(labelleft=False)

    # ── Track 3: FCM_Vsh filled ──────────────────────────────────────────
    ax3 = fig.add_subplot(gs[0, 2], sharey=ax1)
    vsh = wdf['FCM_Vsh'].values
    vsh_gr = wdf.get('Vsh_GR', pd.Series(np.zeros(len(wdf)))).values

    # Colour-fill by Vsh value (green=clean, red=shale)
    from matplotlib.collections import LineCollection
    points = np.array([vsh, depth]).T.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)
    lc = LineCollection(segments, cmap='RdYlGn_r', linewidth=2.0)
    lc.set_array(vsh)
    lc.set_clim(0, 1)
    ax3.add_collection(lc)
    ax3.fill_betweenx(depth, 0, vsh, alpha=0.35,
                      color='salmon', label='FCM_Vsh')
    ax3.plot(vsh_gr, depth, 'k--', lw=0.8, alpha=0.6, label='Vsh_GR')
    ax3.axvline(0.5, color='red', ls=':', lw=1.2, label='NTG cut-off')
    ax3.set_xlim(0, 1.05)
    ax3.set_xlabel('Vsh', fontsize=9)
    ax3.set_title('FCM Vsh', fontsize=9, fontweight='bold')
    ax3.legend(fontsize=7, loc='upper right')
    ax3.grid(axis='x', alpha=0.25)
    ax3.tick_params(labelleft=False)
    plt.colorbar(lc, ax=ax3, label='FCM_Vsh', shrink=0.5, pad=0.01)

    # ── Track 4: FCM hard label colour strip ─────────────────────────────
    ax4 = fig.add_subplot(gs[0, 3], sharey=ax1)
    fcm_lbl = wdf['FCM_label'].values
    for i in range(len(depth) - 1):
        colour = CLUSTER_CMAP(fcm_lbl[i] / max(C_OPTIMAL - 1, 1))
        ax4.fill_betweenx([depth[i], depth[i+1]], 0, 1, color=colour)
    ax4.set_xlim(0, 1)
    ax4.set_xticks([])
    ax4.set_title('FCM\nCluster', fontsize=9, fontweight='bold')
    ax4.tick_params(labelleft=False)

    # Dominant lithology title
    if 'LITH_LABEL' in wdf.columns:
        dom_lith = wdf['LITH_LABEL'].mode().iloc[0]
        subtitle = f'Dominant: {dom_lith}'
    else:
        subtitle = ''

    fig.suptitle(
        f'Well: {well_name}  |  {subtitle}\n'
        f'4-track FCM log panel  (GR  |  RHOB+NPHI  |  FCM Vsh  |  FCM Cluster)',
        fontsize=11, fontweight='bold'
    )
    plt.tight_layout(rect=[0, 0, 1, 0.96])

    safe = well_name.replace('/', '_').replace(' ', '_')
    path = os.path.join(OUT_FIGS, f'log_panel_{safe}.png')
    plt.savefig(path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f'  ✓ Saved: {path}')

print('Generating 4-track log panels …')
for well in rep_wells:
    wdf = master[master['WELL'] == well].copy()
    plot_log_panel(wdf, well)

Representative wells selected: ['25/2-7', '29/6-1', '25/7-2', '16/4-1']
Generating 4-track log panels …
  ✓ Saved: ./outputs/figures/log_panel_25_2-7.png
  ✓ Saved: ./outputs/figures/log_panel_29_6-1.png
  ✓ Saved: ./outputs/figures/log_panel_25_7-2.png
  ✓ Saved: ./outputs/figures/log_panel_16_4-1.png


In [15]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 3 — Transition Zone Identification & Ranked Table
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# WHY define transition zones as max_membership < 0.6:
# A sample with max membership 0.95 is effectively hard-classified already —
# the fuzzy information adds little beyond what K-Means would give.
# A sample with max membership 0.45 has no dominant cluster: it genuinely
# belongs to multiple facies simultaneously.  This is the gradational
# contact zone — the physically interesting interval.
# Threshold 0.6 is conventional (Bezdek, 1981; Cuddy & Glover, 2002).

TRANS_THRESHOLD = 0.6   # max membership below this = transition zone

rows = []
for well, wdf in master.groupby('WELL'):
    n_total = len(wdf)
    n_trans = int((wdf['max_mem'] < TRANS_THRESHOLD).sum())
    pct     = 100.0 * n_trans / n_total if n_total > 0 else 0.0
    # UPDATED (uses calibrated Sand_prob cutoff):
    ntg_fcm_well = float((wdf['Sand_prob'] >= 0.06).sum() / n_total)
    dom_lith = wdf['LITH_LABEL'].mode().iloc[0] \
               if 'LITH_LABEL' in wdf.columns and not wdf['LITH_LABEL'].isna().all() \
               else 'N/A'
    rows.append({
        'Well':               well,
        'N_depth_samples':    n_total,
        'N_transition':       n_trans,
        'Transition_%':       round(pct, 1),
        'NTG_FCM':            round(ntg_fcm_well, 3),
        'Dominant_lith':      dom_lith,
    })

trans_df = pd.DataFrame(rows).sort_values('Transition_%', ascending=False)
trans_df = trans_df.reset_index(drop=True)

print('=' * 72)
print(f'TRANSITION ZONE TABLE  (max cluster membership < {TRANS_THRESHOLD})')
print('=' * 72)
print(trans_df.to_string(index=False))
print()

overall_trans = trans_df['N_transition'].sum() / trans_df['N_depth_samples'].sum() * 100
print(f'Overall transition zone fraction: {overall_trans:.1f}% of all depth intervals')
print()
print('─' * 72)
print('Key finding:')
print(f'  "FCM identified {overall_trans:.0f}% of log intervals as gradational')
print( '   transition zones undetectable by hard lithofacies classifiers"')
print('─' * 72)

# Save
trans_df.to_csv(os.path.join(OUT_DATA, 'transition_zones_per_well.csv'), index=False)
print(f'✓ Saved: transition_zones_per_well.csv')

TRANSITION ZONE TABLE  (max cluster membership < 0.6)
       Well  N_depth_samples  N_transition  Transition_%  NTG_FCM   Dominant_lith
      7/1-1            11359         11357         100.0    0.887           Shale
   25/8-5 S            14502         14468          99.8    0.711           Shale
     35/8-4             2709          2698          99.6    1.000           Shale
    34/7-21            12424         12312          99.1    0.587           Shale
    16/10-2             2437          2411          98.9    0.993       Sandstone
     31/6-5            11098         10948          98.6    0.539           Shale
     34/2-4            23387         23030          98.5    0.849 Sandstone/Shale
   35/11-12            10245         10061          98.2    0.983           Shale
   34/10-35            20614         20206          98.0    0.989 Sandstone/Shale
     16/1-2             1734          1696          97.8    0.984       Sandstone
     25/8-7             8644          8450  